In [1]:
from pathlib import Path

from hakken_ml_toolkit.ml_utils import DSVUtils

In [2]:
version = "0.4.0"
root_path = Path(f"/home/pablo.sanchez2/Documents/GitHub/data/hakken_bio/pubtator3-v{version}")

file_path = root_path / "edges.tsv"

In [3]:
facts_df = DSVUtils.read_dsv(
    file_path,
    delimiter="\t",
    header=0,
)

2025-09-22 10:35:17.321 | INFO     | hakken_ml_toolkit.ml_utils.dsv:read_dsv:46 - Loading DSV file: /home/pablo.sanchez2/Documents/GitHub/data/hakken_bio/pubtator3-v0.4.0/edges.tsv


In [4]:
facts_df.head()

,subject_id,relation_type,object_id,year,year_occurrences,subject_domain,object_domain,subject_id_raw,object_id_raw,number_of_occurrences
0,0003fb22e2049da4ab2c993efa06f726,ASSOCIATE,2a57ef5e287a4d2b1bc3cb68a39bf1f9,2014,2014,GENE,GENE,12400,12842,1
1,0003fb22e2049da4ab2c993efa06f726,ASSOCIATE,87a39259584f0df9790286fa4ce05700,1994,1994|2015,GENE,GENE,12400,17523,2
2,0003fb22e2049da4ab2c993efa06f726,NEGATIVE_CORRELATE,f229d8118bdbd64429226bdc28f03f3f,2018,2018,GENE,GENE,12400,54199,1
3,0004388cac249917f776baf7bcd4471d,PREVENT,38720b76d83a57a04d5b763dc0863436,2009,2009,DNAMUTATION,DISEASE,HGVS:c.1527_1528del;CorrespondingGene:100133941,MESH:D006509,1
4,00048b289b2a5de15aa7833e5fc9340a,NEGATIVE_CORRELATE,a4a1037a390964cd8ae630808fec7b63,2017,2017,GENE,GENE,14313,16323,1


In [10]:
import pandas as pd

entity_cols = ["subject_id", "object_id"]

all_entities = pd.concat([facts_df["subject_id"], facts_df["object_id"]]).unique()
entity_id_mapping = {entity: idx for idx, entity in enumerate(all_entities)}

# Map the IDs to their numerical counterparts for both subject_id and object_id
facts_df["subject_id"] = facts_df["subject_id"].map(entity_id_mapping)
facts_df["object_id"] = facts_df["object_id"].map(entity_id_mapping)

In [11]:
facts_df.head()

,subject_id,relation_type,object_id,year,year_occurrences,subject_domain,object_domain,subject_id_raw,object_id_raw,number_of_occurrences
0,0,ASSOCIATE,5012,2014,2014,GENE,GENE,12400,12842,1
1,0,ASSOCIATE,15539,1994,1994|2015,GENE,GENE,12400,17523,2
2,0,NEGATIVE_CORRELATE,111080,2018,2018,GENE,GENE,12400,54199,1
3,1,PREVENT,6647,2009,2009,DNAMUTATION,DISEASE,HGVS:c.1527_1528del;CorrespondingGene:100133941,MESH:D006509,1
4,2,NEGATIVE_CORRELATE,91375,2017,2017,GENE,GENE,14313,16323,1


In [12]:
min_year = facts_df["year"].min()
max_year = facts_df["year"].max()
print(f"{min_year} {max_year}")

1781 2026


In [13]:
cutoff_year = 2023

In [14]:
import numpy as np

cond = facts_df["year"].between(min_year, cutoff_year)
train_facts_df = facts_df.loc[cond]
train_entities = np.unique(train_facts_df[entity_cols].values)
len(train_entities)

548490

In [15]:
train_entities[0]

np.int64(0)

In [16]:
train_facts_df["year"].max()

np.int64(2023)

In [17]:
import numpy as np

entity_cols = ["subject_id", "object_id"]
cond = facts_df["year"].between(cutoff_year + 1, 2026)
valid_facts_df = facts_df.loc[cond]
valid_entities = np.unique(valid_facts_df[entity_cols].values)
len(valid_entities)

112407

In [18]:
entities_not_in_train = np.setdiff1d(valid_entities, train_entities)

In [19]:
len(entities_not_in_train)

31944

In [24]:
from sklearn.model_selection import train_test_split

# Split the data into training (90%) and validation (10%)
train_df, valid_df = train_test_split(facts_df, test_size=0.1, random_state=42)

# Check the size of each split
print(f"Training set size: {len(train_df)}")
print(f"Validation set size: {len(valid_df)}")

Training set size: 10358022
Validation set size: 1150892


In [35]:
train_entities = np.unique(train_df[entity_cols].values)

valid_entities = np.unique(valid_df[entity_cols].values)
entities_not_in_train = np.setdiff1d(valid_entities, train_entities)
len(entities_not_in_train)

24936

In [36]:
entities_not_in_train

array([   816,    949,   1099, ..., 580415, 580419, 580424],
      shape=(24936,))

In [37]:
valid_df_filtered = valid_df[~valid_df[entity_cols].isin(entities_not_in_train).any(axis=1)]
print(f"Validation set size: {len(valid_df_filtered)}")

Validation set size: 1125057


In [38]:
valid_df.shape

(1150892, 10)

In [34]:
valid_df_filtered.shape

(1125057, 10)